# 00 — Análise do alvo: o gate empírico

Este notebook é **ponto de decisão bloqueante**. As trilhas de modelagem não
devem rodar antes de ele estar executado e com a seção de decisão preenchida.

Ele existe porque quatro escolhas do projeto foram feitas por hipótese e precisam
de evidência antes de virarem compromisso. Ver `docs/03-decisoes.md`, D-09 e D-10.

| # | Pergunta | Decisão que depende dela |
|---|---|---|
| 1 | Equipamentos é mesmo o melhor alvo? | D-01 |
| 2 | As chaves naturais declaradas são únicas? | a semântica de `alterada` em `src/changes.py` |
| 3 | A densidade anual é suficiente? | D-04 e D-10 |
| 4 | As coordenadas dão para a trilha geográfica? | viabilidade da trilha 3 |
| 5 | Que colunas o filtro empírico rejeita nos nove snapshots? | D-06 |

Cada seção termina com um **veredito** escrito. Um número sem veredito não fecha
a decisão.

In [ ]:
import sys
from pathlib import Path

BASE_DIR = Path.cwd().parent
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import changes, schema
from src.graph import MUNICIPIO_SAO_PAULO
from src.paths import PRIMARY_FOLDER

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

PERIODOS = changes.periodos_disponiveis()
SP = MUNICIPIO_SAO_PAULO
con = duckdb.connect()

print(f"Snapshots na camada primária: {PERIODOS}")
print(f"Transições: {[str(t) for t in changes.transicoes(PERIODOS)]}")
print(f"Tabelas no escopo: {len(schema.FACT_TABLES)}")
assert len(PERIODOS) >= 2, "rode `python -m src.pipeline` antes deste notebook"

## 1. Qual tabela é o melhor alvo?

A recomendação de D-01 é `rlEstabEquipamento`, com base em duas competências.
Aqui a comparação é feita sobre a série inteira e **dentro do recorte de São
Paulo**, que é a amostra de fato — os números de D-01 eram nacionais, e um
rótulo pode ser denso no país e esparso no município.

Quatro candidatas, escolhidas por serem tabelas de fato ligadas ao
estabelecimento com quantidade ou classificação variável no tempo.

In [ ]:
CANDIDATAS = {
    "rlEstabEquipamento": "co_equipamento",
    "rlEstabComplementar": "co_leito",
    "rlEstabServClass": "co_servico_especializado",
    "rlEstabInstFisiAssist": "co_instalacao",
}

def cobertura(tabela: str, col_item: str) -> pd.DataFrame:
    linhas = []
    for periodo in PERIODOS:
        fato = PRIMARY_FOLDER / periodo / f"{tabela}.parquet"
        raiz = PRIMARY_FOLDER / periodo / "tbEstabelecimento.parquet"
        if not fato.exists() or not raiz.exists():
            continue
        colunas = {r[0] for r in con.execute(
            f"DESCRIBE SELECT * FROM read_parquet('{fato}')").fetchall()}
        if col_item not in colunas:
            linhas.append({"periodo": periodo, "erro": f"sem coluna {col_item}"})
            continue
        linhas.append(con.execute(f'''
            WITH sp AS (
                SELECT DISTINCT co_unidade FROM read_parquet('{raiz}')
                WHERE co_municipio_gestor = '{SP}'
            )
            SELECT '{periodo}' AS periodo,
                   (SELECT COUNT(*) FROM sp) AS estab_sp,
                   COUNT(*) AS linhas_sp,
                   COUNT(DISTINCT f.co_unidade) AS estab_com_registro,
                   COUNT(DISTINCT f."{col_item}") AS itens_distintos
            FROM read_parquet('{fato}') f JOIN sp USING (co_unidade)
        ''').df().iloc[0].to_dict())
    df = pd.DataFrame(linhas)
    if "estab_com_registro" in df:
        df["cobertura_%"] = (100 * df["estab_com_registro"] / df["estab_sp"]).round(1)
    return df

for tabela, col in CANDIDATAS.items():
    print(f"\n{'=' * 72}\n{tabela}  (item = {col})\n{'=' * 72}")
    print(cobertura(tabela, col).to_string(index=False))

### O espaço de rótulos de cada candidata

Cobertura alta não basta. O que decide é quantos **eventos de aquisição** cada
candidata gera, porque é isso que o modelo tem para aprender, e qual é a
prevalência resultante — uma prevalência muito baixa torna a tarefa
estatisticamente frágil mesmo com muitos eventos absolutos.

In [ ]:
def espaco_de_rotulos(tabela: str, col_item: str) -> pd.DataFrame:
    linhas = []
    for t in changes.transicoes(PERIODOS):
        a = PRIMARY_FOLDER / t.origem / f"{tabela}.parquet"
        b = PRIMARY_FOLDER / t.destino / f"{tabela}.parquet"
        raiz = PRIMARY_FOLDER / t.origem / "tbEstabelecimento.parquet"
        if not (a.exists() and b.exists() and raiz.exists()):
            continue
        try:
            linhas.append(con.execute(f'''
                WITH sp AS (
                    SELECT DISTINCT co_unidade FROM read_parquet('{raiz}')
                    WHERE co_municipio_gestor = '{SP}'
                ),
                itens AS (
                    SELECT DISTINCT "{col_item}" FROM read_parquet('{b}')
                    WHERE "{col_item}" IS NOT NULL
                ),
                tinha AS (
                    SELECT DISTINCT co_unidade, "{col_item}"
                    FROM read_parquet('{a}') JOIN sp USING (co_unidade)
                ),
                tem AS (
                    SELECT DISTINCT co_unidade, "{col_item}"
                    FROM read_parquet('{b}') JOIN sp USING (co_unidade)
                ),
                candidatos AS (
                    SELECT s.co_unidade, i."{col_item}"
                    FROM sp s CROSS JOIN itens i
                    EXCEPT SELECT * FROM tinha
                )
                SELECT '{t.destino}' AS transicao,
                       (SELECT COUNT(*) FROM itens) AS itens,
                       COUNT(*) AS candidatos,
                       COUNT(m.co_unidade) AS aquisicoes
                FROM candidatos c
                LEFT JOIN tem m USING (co_unidade, "{col_item}")
            ''').df().iloc[0].to_dict())
        except Exception as e:
            linhas.append({"transicao": t.destino, "erro": str(e)[:80]})
    df = pd.DataFrame(linhas)
    if "aquisicoes" in df:
        df["prevalencia_%"] = (100 * df["aquisicoes"] / df["candidatos"]).round(3)
    return df

resumos = {}
for tabela, col in CANDIDATAS.items():
    df = espaco_de_rotulos(tabela, col)
    resumos[tabela] = df
    print(f"\n{'=' * 72}\n{tabela}\n{'=' * 72}")
    print(df.to_string(index=False))

In [ ]:
# Comparação lado a lado: total de aquisições e prevalência mediana
comparacao = pd.DataFrame([
    {
        "tabela": tabela,
        "aquisicoes_totais": int(df["aquisicoes"].sum()),
        "candidatos_totais": int(df["candidatos"].sum()),
        "prevalencia_mediana_%": round(float(df["prevalencia_%"].median()), 3),
        "itens": int(df["itens"].max()),
    }
    for tabela, df in resumos.items()
    if "aquisicoes" in df and not df.empty
]).sort_values("aquisicoes_totais", ascending=False)

print(comparacao.to_string(index=False))

> **Veredito 1 — alvo. `rlEstabEquipamento`, confirmado com folga.**
>
> Aquisições somadas nas oito transições, dentro de São Paulo:
>
> | Tabela | Itens | Candidatos | Aquisições | Prevalência mediana |
> |---|---|---|---|---|
> | `rlEstabEquipamento` | 99 | 18.473.151 | **12.081** | 0,065% |
> | `rlEstabServClass` | 72 | 13.919.717 | 8.992 | 0,062% |
> | `rlEstabInstFisiAssist` | 51 | 9.350.614 | 7.185 | 0,087% |
> | `rlEstabComplementar` | 69 | 13.536.992 | **688** | 0,005% |
>
> D-01 se confirma por margem maior que a evidência nacional sugeria. Leitos
> rendem cerca de 86 eventos por transição, o que não sustenta treino nem
> avaliação. `rlEstabInstFisiAssist` fica registrada como alternativa, por ter a
> maior prevalência. Registrado em D-18.

## 2. As chaves naturais declaradas são únicas?

`docs/01-selecao-tabelas.md` declara chave natural para `rlEstabEquipamento` e
`rlEstabComplementar` como **hipótese derivada do dicionário**, nunca verificada.

Isso não é detalhe: sem chave única, `src/changes.py` não distingue uma
modificação de uma remoção seguida de inserção, e a taxa de mudança sai
inflada — cada alteração conta duas vezes.

In [ ]:
def unicidade(tabela: str) -> pd.DataFrame:
    chave = schema.CNES_NATURAL_KEY.get(tabela)
    if not chave:
        return pd.DataFrame([{"tabela": tabela, "chave": None,
                              "obs": "sem chave natural declarada"}])
    cols = ", ".join(f'"{c}"' for c in chave)
    linhas = []
    for periodo in PERIODOS:
        p = PRIMARY_FOLDER / periodo / f"{tabela}.parquet"
        if not p.exists():
            continue
        r = con.execute(f'''
            SELECT COUNT(*) AS linhas,
                   COUNT(*) - COUNT(DISTINCT ({cols})) AS duplicadas
            FROM read_parquet('{p}')
        ''').df().iloc[0]
        linhas.append({
            "tabela": tabela, "periodo": periodo,
            "chave": " + ".join(chave),
            "linhas": int(r["linhas"]), "duplicadas": int(r["duplicadas"]),
            "duplicadas_%": round(100 * r["duplicadas"] / max(r["linhas"], 1), 3),
        })
    return pd.DataFrame(linhas)

for tabela in schema.CNES_NATURAL_KEY:
    print(unicidade(tabela).to_string(index=False), "\n")

In [ ]:
# Se houver duplicadas, que colunas faltam na chave? Testa acrescentar cada
# coluna candidata e mede o quanto ela reduz a duplicidade.
def chave_minima(tabela: str, periodo: str) -> pd.DataFrame:
    p = PRIMARY_FOLDER / periodo / f"{tabela}.parquet"
    if not p.exists():
        return pd.DataFrame()
    base = list(schema.CNES_NATURAL_KEY.get(tabela, ()))
    todas = [c for c in schema.CNES_EXTRACT_COLUMNS[tabela]
             if not c.startswith("to_char")]
    linhas = []
    for extra in [None] + [c for c in todas if c not in base]:
        cols = base + ([extra] if extra else [])
        if not cols:
            continue
        expr = ", ".join(f'"{c}"' for c in cols)
        r = con.execute(f'''SELECT COUNT(*) - COUNT(DISTINCT ({expr})) AS dup
                            FROM read_parquet('{p}')''').fetchone()[0]
        linhas.append({"acrescentando": extra or "(chave declarada)",
                       "duplicadas": int(r)})
    return pd.DataFrame(linhas).sort_values("duplicadas")

for tabela in schema.CNES_NATURAL_KEY:
    print(f"\n{tabela} em {PERIODOS[-1]}:")
    print(chave_minima(tabela, PERIODOS[-1]).to_string(index=False))

> **Veredito 2 — chaves naturais. As duas hipóteses estavam certas.**
>
> Zero duplicatas em 201701 e 202501, para
> `rlEstabEquipamento` por (`co_unidade`, `co_equipamento`, `co_tipo_equipamento`,
> `tp_sus`) e `rlEstabComplementar` por (`co_unidade`, `co_leito`,
> `co_tipo_leito`).
>
> Deixam de ser hipóteses derivadas do dicionário e passam a fato verificado. A
> classificação `alterada` de `src/changes.py` é confiável para essas duas
> tabelas — e só para elas.

## 3. A densidade anual é suficiente?

D-04 fixou nove snapshots anuais provisoriamente. A pergunta de D-10 é se um ano
de intervalo esconde ciclos relevantes.

O sinal a olhar: se a taxa de mudança anual for muito alta, o intervalo está
agregando eventos que não se pode separar, e a série precisa densificar. Se for
baixa, anual já é fino demais e o custo do ETL não se paga.

In [ ]:
# Exige que `python -m src.changes` tenha rodado.
try:
    taxa = changes.taxa_de_mudanca()
except FileNotFoundError as e:
    print(e)
    taxa = None

if taxa is not None:
    foco = taxa[taxa["tabela"].isin(CANDIDATAS)]
    print(foco[["tabela", "periodo_destino", "linhas_origem", "linhas_destino",
                "inserida", "removida", "alterada", "taxa_mudanca",
                "chave_declarada"]].to_string(index=False))

In [ ]:
if taxa is not None and not taxa.empty:
    fig, ax = plt.subplots(figsize=(11, 4.5))
    for tabela, grupo in taxa[taxa["tabela"].isin(CANDIDATAS)].groupby("tabela"):
        ax.plot(grupo["periodo_destino"], grupo["taxa_mudanca"],
                marker="o", label=tabela)
    ax.set_title("Taxa de mudança anual por tabela candidata")
    ax.set_xlabel("transição (período de destino)")
    ax.set_ylabel("eventos / linhas")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    print("\nAtenção: onde chave_declarada é False, cada modificação conta como")
    print("remoção + inserção e a taxa está superestimada.")

> **Veredito 3 — densidade de snapshots. Anual mantida.**
>
> Taxa de mudança de `rlEstabEquipamento` nas oito transições: 0,112, 0,094,
> 0,101, 0,100, 0,095, 0,085, 0,082, 0,089. Mediana 0,094, amplitude inteira
> entre 0,082 e 0,112, sem pico nas transições de pandemia.
>
> Série plana é justamente o que indica que o intervalo não está agregando
> eventos que se queira separar. Não há evidência de ciclo escondido que
> justifique o custo de densificar. D-10 fechada.
>
> Leitura: `tbEstabelecimento` aparece com taxa acima de 1,0 porque não tem chave
> natural declarada — cada modificação conta duas vezes e `alterada` sai zerada.

## 4. A trilha geográfica é viável?

A trilha 3 depende inteiramente de `nu_latitude` e `nu_longitude`, cujo
preenchimento **nunca foi verificado**. Uma medição preliminar em 201701 achou
cobertura de cerca de 0,5% em São Paulo, o que ameaça a trilha inteira. Aqui a
cobertura é medida em todos os snapshots, para ver se ela melhora ao longo do
tempo.

In [ ]:
def cobertura_geografica() -> pd.DataFrame:
    linhas = []
    for periodo in PERIODOS:
        p = PRIMARY_FOLDER / periodo / "tbEstabelecimento.parquet"
        if not p.exists():
            continue
        linhas.append(con.execute(f'''
            SELECT '{periodo}' AS periodo,
                   COUNT(*) AS estab_sp,
                   COUNT(nu_latitude) AS com_coordenada,
                   SUM(CASE WHEN nu_latitude BETWEEN -34 AND 6
                             AND nu_longitude BETWEEN -74 AND -34
                             AND NOT (nu_latitude = 0 AND nu_longitude = 0)
                            THEN 1 ELSE 0 END) AS coordenada_plausivel
            FROM read_parquet('{p}')
            WHERE co_municipio_gestor = '{SP}'
        ''').df().iloc[0].to_dict())
    df = pd.DataFrame(linhas)
    df["cobertura_%"] = (100 * df["com_coordenada"] / df["estab_sp"]).round(2)
    df["plausivel_%"] = (100 * df["coordenada_plausivel"] / df["estab_sp"]).round(2)
    return df

geo = cobertura_geografica()
print(geo.to_string(index=False))

In [ ]:
# Cobertura acumulada: um estabelecimento com coordenada em QUALQUER snapshot
# pode ser posicionado, porque endereço muda pouco. É o limite superior real
# da trilha geográfica.
arquivos = [str(PRIMARY_FOLDER / p / "tbEstabelecimento.parquet")
            for p in PERIODOS
            if (PRIMARY_FOLDER / p / "tbEstabelecimento.parquet").exists()]
lista = ", ".join(f"'{a}'" for a in arquivos)

acumulada = con.execute(f'''
    WITH sp AS (
        SELECT co_unidade, nu_latitude, nu_longitude
        FROM read_parquet([{lista}])
        WHERE co_municipio_gestor = '{SP}'
    )
    SELECT COUNT(DISTINCT co_unidade) AS estab_sp_total,
           COUNT(DISTINCT CASE WHEN nu_latitude BETWEEN -34 AND 6
                                AND nu_longitude BETWEEN -74 AND -34
                                AND NOT (nu_latitude = 0 AND nu_longitude = 0)
                               THEN co_unidade END) AS posicionaveis
    FROM sp
''').df()
acumulada["cobertura_acumulada_%"] = (
    100 * acumulada["posicionaveis"] / acumulada["estab_sp_total"]).round(2)
print(acumulada.to_string(index=False))

In [ ]:
if geo["plausivel_%"].max() > 5:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(geo["periodo"], geo["plausivel_%"])
    ax.set_title("Cobertura de coordenada plausível em São Paulo, por snapshot")
    ax.set_ylabel("% dos estabelecimentos")
    ax.grid(alpha=0.3, axis="y")
    plt.tight_layout()
    plt.show()
else:
    print("Cobertura baixa em todos os snapshots — gráfico omitido.")
    print("A trilha geográfica precisa de fonte externa de geocodificação,")
    print("ou de ser restrita ao subconjunto posicionável, com a ressalva de")
    print("que esse subconjunto não é aleatório.")

> **Veredito 4 — trilha geográfica. Teto estrutural de 57%.**
>
> Cobertura cresce de 0,45% (2017) a 57,5% (2022), mas a união de todos os
> snapshots dá 57,3% — quem não tem coordenada em 2022 nunca teve. O teto não é
> temporal, é estrutural, e 43% dos estabelecimentos jamais serão nós.
>
> Resolvido antes deste gate: D-15 trata posição como invariante no tempo, D-17
> mede o teto e rejeita CEP-5 como substituto (raio mediano 4,90 km contra 8,70
> km de um controle embaralhado — informativo, mas só ~2x melhor que o azar).
>
> Pendente: caracterizar **quem** fica de fora, no
> `notebook/04_recorte_e_dados_externos.ipynb`.

## 5. O filtro empírico sobre os nove snapshots

D-06 aplicou o filtro empírico usando apenas as competências 201701 e 202501, do
relatório antigo, e ele rejeitou uma única coluna. Aqui o filtro é reaplicado
sobre a série inteira: uma coluna é degenerada se está 100% nula, ou constante,
em **todos** os snapshots medidos.

In [ ]:
def triagem_empirica() -> pd.DataFrame:
    linhas = []
    for tabela, colunas in schema.CNES_EXTRACT_COLUMNS.items():
        arquivos = [PRIMARY_FOLDER / p / f"{tabela}.parquet" for p in PERIODOS]
        arquivos = [a for a in arquivos if a.exists()]
        if not arquivos:
            continue
        lista = ", ".join(f"'{a}'" for a in arquivos)
        presentes = {r[0] for r in con.execute(
            f"DESCRIBE SELECT * FROM read_parquet([{lista}])").fetchall()}
        for coluna in colunas:
            if coluna not in presentes:
                linhas.append({"tabela": tabela, "coluna": coluna,
                               "motivo": "ausente do Parquet"})
                continue
            r = con.execute(f'''
                SELECT COUNT(*) AS n,
                       COUNT("{coluna}") AS nao_nulos,
                       COUNT(DISTINCT "{coluna}") AS distintos
                FROM read_parquet([{lista}])
            ''').df().iloc[0]
            motivo = None
            if r["n"] == 0:
                motivo = "tabela vazia em todos os snapshots"
            elif r["nao_nulos"] == 0:
                motivo = "100% nula em todos os snapshots"
            elif r["distintos"] <= 1:
                motivo = "constante em todos os snapshots"
            if motivo:
                linhas.append({"tabela": tabela, "coluna": coluna, "motivo": motivo})
    return pd.DataFrame(linhas)

rejeitadas = triagem_empirica()
print(f"{len(rejeitadas)} colunas hoje `util` que o filtro empírico rejeita "
      f"sobre {len(PERIODOS)} snapshots:\n")
if not rejeitadas.empty:
    print(rejeitadas.sort_values(["motivo", "tabela"]).to_string(index=False))

> **Veredito 5 — filtro empírico. Uma rejeição.**
>
> `rlEstabUnidAcolhim.tp_sus_nao_sus`, constante em toda a série, reclassificada
> para `descartada`. Colunas `util`: 389 para 388.
>
> O resultado importa mais pelo que **não** mudou: ampliar de duas para nove
> competências acrescentou uma única rejeição. O crivo de D-06 é estável, não
> aperta indefinidamente conforme se olha mais dado.
>
> **Achado colateral, virou D-20.** A leitura conjunta dos nove snapshots falhou:
> três das 44 tabelas têm colunas que somem e voltam, e o padrão aponta 201901
> como competência anômala. Leitura direta de vários Parquet via DuckDB precisa
> de `union_by_name=true`.

## Decisão final

Todos os cinco vereditos fechados. Registrado em `docs/03-decisoes.md`, D-18 a D-20.

| Decisão | Status | Valor fixado | Evidência |
|---|---|---|---|
| D-01 alvo | fechada | `rlEstabEquipamento` | 12.081 aquisições contra 688 de leitos |
| D-04 / D-10 densidade | fechada | nove snapshots anuais | taxa entre 0,082 e 0,112, série plana |
| D-06 filtro empírico | fechada | uma rejeição a mais | crivo estável de 2 para 9 competências |
| Chave natural | fechada | as duas hipóteses confirmadas | zero duplicatas |
| Trilha 3 geográfica | fechada com ressalva | teto de 57%, posição invariante | D-15, D-17 |

**Achado que não estava previsto:** a prevalência de 0,065% é severa e
irredutível. Duas restrições do espaço de candidatos foram testadas e rejeitadas
— por par (tipo de unidade, equipamento) corta só 1,5%; por estabelecimento já
equipado corta 50% dos candidatos mas leva 33% dos positivos. MAP@k passa a ser a
métrica de destaque. D-19.

**Consequências aplicadas ao código:**

- `docs/01-selecao-tabelas.md` — `rlEstabUnidAcolhim.tp_sus_nao_sus` descartada
- `docs/02-metodologia.md` — MAP@k promovida a métrica de destaque
- `docs/03-decisoes.md` — D-09 e D-10 fechadas; D-18, D-19 e D-20 acrescentadas
- `src/tasks.py` — nenhuma mudança: os defaults já apontavam para equipamentos